In [2]:
import pandas as pd
import os
import subprocess
from collections import Counter

In [4]:
df_train = pd.read_csv('train_final.csv')

In [40]:
df_train.head()

,user_query,dialogue_context,retrieved_docs,true_q_i,true_a_i,dataset,split,original_id,paraphrase_rank
0,How does T-cell count contribute to the severi...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p1,1
1,What significance does T-cell count hold in th...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p2,2
2,Can elevated or reduced T-cell counts serve as...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p3,3
3,In what way is T-cell count linked to the deve...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p4,4
4,Does the relationship between T-cell count and...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p5,5


In [41]:
# Fill NaN in true_q_i with a placeholder, then map each unique true_q_i to a unique Question_ID
df['true_q_i'] = df['true_q_i'].fillna(-1)

unique_q = df['true_q_i'].unique()
q_to_id = {q: i+1 for i, q in enumerate(sorted(unique_q))}

df['Question_ID'] = df['true_q_i'].map(q_to_id)

# Update original_id: replace first part with new Question_ID
df['original_id'] = df['original_id'].astype(str)
df['original_id'] = df['original_id'].str.replace(r'^\d+', lambda x: str(df.loc[x.start(), 'Question_ID']), regex=True)

# Optional: clean up placeholder if needed
df.loc[df['true_q_i'] == -1, 'true_q_i'] = pd.NA

In [42]:
df['query_id'] = df['Question_ID'].astype(str) + '_' + df['paraphrase_rank'].astype(str)

# Optional: drop old original_id if no longer needed
# df = df.drop(columns=['original_id'])

df[['query_id', 'Question_ID', 'paraphrase_rank', 'user_query']].head()
df.drop(columns=['split'], inplace=True)

In [43]:
df.head(30)

,user_query,dialogue_context,retrieved_docs,true_q_i,true_a_i,dataset,original_id,paraphrase_rank,Question_ID,query_id
0,How does T-cell count contribute to the severi...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,4148_p1,1,4148,4148_1
1,What significance does T-cell count hold in th...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,4148_p2,2,4148,4148_2
2,Can elevated or reduced T-cell counts serve as...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,4148_p3,3,4148,4148_3
3,In what way is T-cell count linked to the deve...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,4148_p4,4,4148,4148_4
4,Does the relationship between T-cell count and...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,4148_p5,5,4148,4148_5
5,Is there evidence to suggest that individuals ...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,4148_p6,6,4148,4148_6
6,What does MVO stand for?,NaN,"[""Title: Gene Knockdowns in Adult Animals: PPM...",What is MVO?,MVO is not defined or mentioned in the provide...,covidqa,4148_p1,1,3304,3304_1
7,Is MVO an abbreviation of some sort?,NaN,"[""Title: Gene Knockdowns in Adult Animals: PPM...",What is MVO?,MVO is not defined or mentioned in the provide...,covidqa,4148_p2,2,3304,3304_2
8,Can you define what MVO means?,NaN,"[""Title: Gene Knockdowns in Adult Animals: PPM...",What is MVO?,MVO is not defined or mentioned in the provide...,covidqa,4148_p3,3,3304,3304_3
9,Do you know the definition of MVO?,NaN,"[""Title: Gene Knockdowns in Adult Animals: PPM...",What is MVO?,MVO is not defined or mentioned in the provide...,covidqa,4148_p4,4,3304,3304_4


In [44]:
stats = (df.groupby('dataset')
         .agg(unique_questions=('Question_ID', 'nunique'),
              total_queries=('user_query', 'count'))
         .reset_index())

total = pd.DataFrame({
    'dataset': ['**Total**'],
    'unique_questions': [stats['unique_questions'].sum()],
    'total_queries': [stats['total_queries'].sum()]
})

print(pd.concat([stats, total], ignore_index=True).to_markdown(index=False))

| dataset   |   unique_questions |   total_queries |
|:----------|-------------------:|----------------:|
| covidqa   |               1468 |            8820 |
| expertqa  |               1824 |           10944 |
| hagrid    |                634 |            3828 |
| hotpotqa  |               2051 |           12306 |
| msmarco   |               2011 |           12066 |
| **Total** |               7988 |           47964 |


In [45]:
df.to_csv('train_paraphrased_updated.csv', index=False)

KeyboardInterrupt: 

In [ ]:
# finalize_train_paraphrased_updated.py
# 为每个 Question_ID 只保留语义最不同的 3 条改写（基于新版数据）
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch

INPUT_FILE = "train_paraphrased_updated.csv"
OUTPUT_FILE = "train_paraphrased_final.csv"

print("加载数据...")
df = pd.read_csv(INPUT_FILE)
print(f"原始改写条数: {len(df):,}")
print(f"原始 Question_ID 数量: {df['Question_ID'].nunique():,}")

# 加载模型
print("\n加载 all-MiniLM-L6-v2 用于多样性选择...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

final_rows = []
print("\n开始处理每个 Question_ID，只保留语义最不同的 3 条改写...")

for qid, group in df.groupby('Question_ID'):
    group = group.reset_index(drop=True)
    n = len(group)
    
    if n <= 3:
        selected = group.copy()
        selected['paraphrase_rank'] = range(1, n + 1)
        final_rows.append(selected)
        continue

    # 编码所有 user_query
    queries = group['user_query'].tolist()
    embs = model.encode(queries, convert_to_tensor=True, normalize_embeddings=True)
    sim_matrix = util.cos_sim(embs, embs)

    selected_idx = []
    remaining = list(range(n))

    # 选第一个（任意）
    first = 0
    selected_idx.append(first)
    remaining.remove(first)

    # 贪心选剩下 2 个最不相似的
    for _ in range(2):
        if not remaining:
            break
        sim_to_selected = sim_matrix[remaining][:, selected_idx].mean(dim=1)
        best_global_idx = remaining[sim_to_selected.argmin().item()]
        selected_idx.append(best_global_idx)
        remaining.remove(best_global_idx)

    selected = group.iloc[selected_idx].copy()
    selected['paraphrase_rank'] = range(1, 4)  # 重新编号 1,2,3
    final_rows.append(selected)

# 合并
final_df = pd.concat(final_rows, ignore_index=True)

# 保持原列顺序（不丢失任何列）
final_df = final_df[df.columns]

# 保存
final_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n处理完成！")
print(f"   最终训练样本数: {len(final_df):,}")
print(f"   最终 Question_ID 数: {final_df['Question_ID'].nunique():,}")
print(f"   每题平均改写数: {len(final_df)/final_df['Question_ID'].nunique():.2f}")
print(f"   每题正好 3 条的 Question_ID 数量: {(final_df['Question_ID'].value_counts() == 3).sum():,}")
print(f"   已保存 → {OUTPUT_FILE}")

c:\Users\24043941r\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


加载数据...
原始改写条数: 47,964
原始 Question_ID 数量: 7,988

加载 all-MiniLM-L6-v2 用于多样性选择...

开始处理每个 Question_ID，只保留语义最不同的 3 条改写...

处理完成！
   最终训练样本数: 23,964
   最终 Question_ID 数: 7,988
   每题平均改写数: 3.00
   每题正好 3 条的 Question_ID 数量: 7,988
   已保存 → train_paraphrased_final.csv


In [47]:
# 重新生成正确的 query_id：Question_ID + "_p" + paraphrase_rank
final_df['query_id'] = final_df['Question_ID'].astype(str) + '_p' + final_df['paraphrase_rank'].astype(str)

# 检查结果
final_df[['Question_ID', 'paraphrase_rank', 'query_id', 'user_query']].head(10)

,Question_ID,paraphrase_rank,query_id,user_query
0,1,1,1_p1,What piano-playing songwriter received two Gra...
1,1,2,1_p2,What renowned composer-pianist earned distinct...
2,1,3,1_p3,Which songwriter won two Grammys for composing...
3,2,1,2_p1,What British musician recorded the hit single ...
4,2,2,2_p2,Which artist who rose to fame after releasing ...
5,2,3,2_p3,Whose 2020 album featured the lead single titl...
6,3,1,3_p1,Who wrote the script for the movie Horrible Bo...
7,3,2,3_p2,Who is a screenwriter that also happens to wri...
8,3,3,3_p3,The screenwriter for the comedy film Horrible ...
9,4,1,4_p1,Who is the singer who collaborated on the song...


In [50]:
final_df.to_csv("backup.csv", index=False)
